# 04 · 重叠样本与论文基础统计

同品种同月份配对，不用作者收益补数据。均值/标准差/极值/分位数以完全相同的月度样本计算，收益单位为百分比、误差为基点；未年化，合并统计不是投资组合收益。

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if (ROOT / 'project' / 'pyproject.toml').exists(): ROOT = ROOT / 'project'
assert (ROOT / 'configs' / 'commodity_map.csv').exists(), 'Open the project folder first'
sys.path.insert(0, str(ROOT / 'src'))
from project.paths import ensure_directories
ensure_directories()
pd.set_option('display.max_columns', 14)


In [ ]:
from project.diagnostics import run_diagnostics,overlap_validation
panel=pd.read_parquet(ROOT/'data/processed/commodity_monthly.parquet')
run_diagnostics(panel,'main')
paired,pooled,errors=overlap_validation(panel)
display(pooled.round(4))
display(errors.round(3))
display(paired.nlargest(15,'abs_error_bp')[['paper_code','date','selected_futcode','entry_price','exit_price','error_bp']])
print('Paired observations:',len(paired),'MAE (bp):',paired.abs_error_bp.mean())
check=pd.read_csv(ROOT/'results/replication/tables/main/internet_II_author_check.csv')
assert check.n_difference.eq(0).all()
assert check.mean_difference.abs().max()<.0051
assert check.sd_difference.abs().max()<.0051
assert check.sr_difference.abs().max()<.0051
assert check.contango_including_flat_difference.abs().max()<.0051


Internet-II的contango频率需要将零斜率计入才能与作者表匹配；策略仍严格按负/正斜率。高相关不等于准确，原作者无逐月合约ID/日价格，仍需追查大误差。Internet-III等定价表尚未复现。结果目录：results/replication/overlap_validation 和 tables/main。